# Main S1 — retrieval (M1 corpus, 7 retrievers, efficiency)
**Run with: Save Version → "Save & Run All (Commit)"**, accelerator **GPU T4 x2**, Internet On. It runs unattended; if any step fails the notebook stops with `FAILED` — send me the log. When it finishes, download `main_s1_results.zip` from the version's Output tab and send it to me.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 1 — hardware, code frozen at 22f161a (branch main-v1), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
sh("git clone --depth 1 --branch main-v1 https://github.com/adeelcheena791/rag-embed-transfer.git rag")
os.chdir("/kaggle/working/rag")
sh("git log --oneline -1")
sh("test \"$(git rev-parse HEAD)\" = 22f161a6e48a188ce8114bc8f6d402bcf7fdf327 || (echo WRONG CODE COMMIT; exit 1)")
sh("pip install -q -r requirements-retrieval-pinned.txt")
sh("python -m src.check_env --lock requirements-retrieval.lock")
sh("python -m src.env_report --session main_S1 --out results/raw/env_main_S1.json")
sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — data: pinned download (P1) + join (expect 2,255 joined of 3,452)
sh("python -m src.data.download --config configs/main.yaml")
sh("python -m src.data.join_nq --config configs/main.yaml")

In [ ]:
# Cell 3 — M1 sub-corpus: BM25 over the full corpus for ALL 3,452 judged queries, then 300,000 passages
sh("python -m src.retrieval.bm25 --config configs/main.yaml --corpus full --queries judged")
sh("python -m src.data.subcorpus --config configs/main.yaml --size 300000 --population judged --name main")

In [ ]:
# Cell 4 — R0: BM25 on the main corpus, all judged queries, with CPU latency
sh("python -m src.retrieval.bm25 --config configs/main.yaml --corpus main --queries judged --latency")

In [ ]:
# Cell 5 — efficiency (M3): GPU encoding throughput + batch-1 GPU query latency, all 6 dense models (~35 min)
sh("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/main.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128")

In [ ]:
# Cell 6 — encode the 300k main corpus + all judged queries with all 6 dense models (~2.5 h)
sh("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/main.yaml --models R1,R2,R3,R5,R4,R6 --corpus main")

In [ ]:
# Cell 7 — exact search (CPU) for all judged queries, with search latency
sh("python -m src.retrieval.search --config configs/main.yaml --model R1 --corpus main --queries judged --latency")
sh("python -m src.retrieval.search --config configs/main.yaml --model R2 --corpus main --queries judged --latency")
sh("python -m src.retrieval.search --config configs/main.yaml --model R3 --corpus main --queries judged --latency")
sh("python -m src.retrieval.search --config configs/main.yaml --model R5 --corpus main --queries judged --latency")
sh("python -m src.retrieval.search --config configs/main.yaml --model R4 --corpus main --queries judged --latency")
sh("python -m src.retrieval.search --config configs/main.yaml --model R6 --corpus main --queries judged --latency")

In [ ]:
# Cell 8 — efficiency (M4): CPU encoding throughput + batch-1 CPU query latency on THIS session's CPU (~1 h)
sh("python -m src.retrieval.throughput --config configs/main.yaml --models R1,R2,R3,R5,R4,R6 --n 2000 --batch 128 --device cpu")

In [ ]:
# Last cell — summary + package (becomes this version's saved output)
sh("python -m src.pilot_summary --config configs/main.yaml")
sh("zip -qr /kaggle/working/main_s1_results.zip results data/processed configs/resolved_revisions.json logs requirements-*.lock data/raw/nq_qrels/test.tsv data/raw/nq_corpus/queries")
sh("ls -la /kaggle/working/main_s1_results.zip")
sh("rm -rf /kaggle/working/rag/data/raw /tmp/models")   # re-downloadable; keeps the saved output small